In [ ]:
!git clone https://github.com/NVIDIA/apex
!pip install optuna==2.3.0
!pip install transformers==4.33.1
!pip install farasapy
!pip install pyarabic
!pip install sklearn
!pip install accelerate -U
!pip install  scikit-learn
!pip install tensorflow_gpu==2.10.0
!git clone https://github.com/aub-mind/arabert
!git clone https://huggingface.co/aubmindlab/bert-large-arabertv02
!pip install pandas
!pip install bitsandbytes
!pip install accelerate 
!pip install chardet
!pip install deep_translator
conda install pytorch==2.0.0 torchvision==0.15.0 torchaudio==2.0.0 pytorch-cuda=11.2 -c pytorch -c nvidia

In [ ]:
import pandas as pd
import numpy as np
import os.path
from arabert.preprocess import ArabertPreprocessor
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, precision_score , recall_score

from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, BertTokenizer
from transformers.data.processors import SingleSentenceClassificationProcessor
from transformers import Trainer , TrainingArguments
from transformers.trainer_utils import EvaluationStrategy
from transformers.data.processors.utils import InputFeatures
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.utils import resample
import logging
import torch
import optuna 
import csv
from sklearn.model_selection import train_test_split
import tensorflow as tf
from accelerate import Accelerator
from torch.utils.data.dataloader import DataLoader
from deep_translator import GoogleTranslator

In [ ]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():    

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
gpu_device = tf.config.list_physical_devices('GPU')[0]
tf.config.experimental.set_memory_growth(gpu_device, True)
tf.keras.backend.set_floatx('float16')

In [ ]:
DATA_COLUMN = 'text'
LABEL_COLUMN = 'label'
df1=#YOUR DATA

train_AJGT, test_AJGT = train_test_split(df1, test_size=0.3,random_state=42)
train_df = pd.DataFrame({
    'id':range(len(train_AJGT)),
    'label':train_AJGT["label"],
    'text': train_AJGT["text"].replace(r'\n', ' ', regex=True),
})

dev_df = pd.DataFrame({
    'id':range(len(test_AJGT)),
    'label':test_AJGT["label"],
    'text': test_AJGT["text"].replace(r'\n', ' ', regex=True),
})

In [ ]:
class BERTDataset(Dataset):
    def __init__(self, text, target, model_name, max_len, label_map):
      super(BERTDataset).__init__()
      self.text = text
      self.target = target
      self.tokenizer_name = model_name
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.max_len = max_len
      self.label_map = label_map
      

    def __len__(self):
      return len(self.text)

    def __getitem__(self,item):
      text = str(self.text[item])
      text = " ".join(text.split())


        
      input_ids = self.tokenizer.encode(
          text,
          add_special_tokens=True,
          max_length=self.max_len,
          truncation='longest_first'
      )     
    
      attention_mask = [1] * len(input_ids)

      # Zero-pad up to the sequence length.
      padding_length = self.max_len - len(input_ids)
      input_ids = input_ids + ([self.tokenizer.pad_token_id] * padding_length)
      attention_mask = attention_mask + ([0] * padding_length)    
      
      return InputFeatures(input_ids=input_ids, attention_mask=attention_mask, label=self.label_map[self.target[item]])

In [ ]:
max_len=#CHANGE TO MAX LENGTH OF TEXT
label_map = { v:index for index, v in enumerate(df1['label'].unique()) }
model_name='bert-large-arabertv02'

train_dataset = BERTDataset(train_df['text'].to_list(),train_df['label'].to_list(),model_name,max_len,label_map)
test_dataset = BERTDataset(dev_df['text'].to_list(),dev_df['label'].to_list(),model_name,max_len,label_map)

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, return_dict=True, num_labels=len(label_map),from_tf=True).to(device)

In [ ]:
training_args = TrainingArguments("./train")
training_args.output_dir='./checkpoint'
training_args = TrainingArguments("./train1")
training_args.evaluate_during_training = True
training_args.do_train=True,
training_args.do_eval=False,
training_args.adam_epsilon = 1e-8
training_args.learning_rate = 0.000001
training_args.fp16 = True
training_args.per_device_train_batch_size = 1024
training_args.per_device_eval_batch_size = 1024
training_args.gradient_accumulation_steps = 2
training_args.num_train_epochs= 4
training_args.gradient_checkpointing=True
training_args.optim="adafactor"
training_args.load_best_model_at_end=True
training_args.predict_with_generate=True

training_args.evaluation_strategy = EvaluationStrategy.EPOCH
training_args.logging_steps = 200
training_args.seed = 42
training_args.disable_tqdm = False
training_args.lr_scheduler_type = 'cosine'

In [ ]:
def compute_metrics(p): #p should be of type EvalPrediction
  preds = np.argmax(p.predictions, axis=1)
  assert len(preds) == len(p.label_ids)
  #print(classification_report(p.label_ids,preds))
  #print(confusion_matrix(p.label_ids,preds))
    
  macro_f1_pos_neg = f1_score(p.label_ids,preds,average='macro',labels=[0,1],zero_division=np.nan)
  macro_f1 = f1_score(p.label_ids,preds,average='macro',zero_division=np.nan)
  macro_precision = precision_score(p.label_ids,preds,average='macro',zero_division=np.nan)
  macro_recall = recall_score(p.label_ids,preds,average='macro',zero_division=np.nan)
  acc = accuracy_score(p.label_ids,preds)
  return {
      'macro_f1' : macro_f1,
      'macro_f1_pos_neg' : macro_f1_pos_neg,  
      'macro_precision': macro_precision,
      'macro_recall': macro_recall,
      'accuracy': acc
  }

In [ ]:
trainer = trainer = Trainer(
            model =  model_init(),
            args = training_args,
            train_dataset = train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
            #optimizers=(adam_bnb_optim, None),
        )

In [ ]:
trainer.train()
trainer.save_model(#PATH TO SAVE MODEL)